# Lab Extension: RAG Evaluation — Applied Use Cases with DeepEval
### 10 Exercises Across Different Domains

| # | Domain | Metric(s) in focus |
|---|--------|---------------------|
| 1 | Telecom customer support | Faithfulness, Answer Relevancy |
| 2 | Legal discovery | Contextual Precision vs Recall trade-off |
| 3 | Medical symptom checker | Custom `GEval` (safety hedging) |
| 4 | Financial advisor bot | `HallucinationMetric` |
| 5 | E-commerce product Q&A | Contextual Relevancy under noise |
| 6 | HR policy assistant | Baseline regression comparison |
| 7 | Code documentation assistant | Faithfulness on technical claims |
| 8 | Travel booking assistant | Answer Relevancy score buckets |
| 9 | Your own domain | End-to-end harness (open-ended) |
| 10 | CI gating | Cost-aware `assert_test` gating |

> Reuse the `judge` object from the main lab notebook if you're running this in the same kernel/session.
> The setup cell below redefines it so this notebook also works standalone.


## Setup (standalone — skip if `judge` already exists in your kernel)

In [ ]:
# DeepEval + LiteLLM. Same setup as the main lab.
!pip install -q -U deepeval litellm python-dotenv


In [ ]:
import os
import logging
from dotenv import load_dotenv
from deepeval.models import LiteLLMModel

logging.getLogger("deepeval").setLevel(logging.ERROR)
os.environ["DEEPEVAL_TELEMETRY"] = "0"
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

load_dotenv()

MODEL_ID = os.getenv("EVAL_JUDGE_MODEL", "openrouter/deepseek/deepseek-chat-v3.1")
judge = LiteLLMModel(model=MODEL_ID, temperature=0)
THRESHOLD = 0.7
print(f"Judge model: {MODEL_ID}")


---
## Exercise 1 — Telecom Customer Support Bot
### Faithfulness + Answer Relevancy

**Use case:** A telecom's support bot answers billing and plan questions using help-center articles as
retrieved context. A hallucinated fee or an off-topic answer both cost the company a support ticket.

**Task:** Complete `evaluate_support_bot`, which should mirror `run_eval_suite` from the main lab: for each
response, measure Faithfulness and Answer Relevancy, and mark it `passed` only if **both** scores clear
`threshold`.


In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    ContextualPrecisionMetric, ContextualRecallMetric, ContextualRelevancyMetric,
    FaithfulnessMetric, AnswerRelevancyMetric, HallucinationMetric, GEval,
)
from deepeval.test_case import LLMTestCaseParams

SUPPORT_ARTICLES = {
    "art_overage_0": "Data overage charges apply once you exceed your monthly plan's data allowance, billed at $10 per extra GB.",
    "art_autopay_0": "Enrolling in AutoPay gives a $5/month discount, applied automatically on your next bill after enrollment.",
    "art_intl_0": "International roaming must be enabled in the app before travel; standard plans do not include roaming by default.",
}

support_responses = [
    {
        "query": "Why was I charged extra this month?",
        "answer": "You were charged extra because you went over your monthly data allowance, at a rate of $10 per additional GB.",
        "retrieval_context": [SUPPORT_ARTICLES["art_overage_0"]],
    },
    {
        "query": "Does AutoPay save me money?",
        "answer": "Yes — AutoPay gives you a $5 monthly discount that starts on your very first bill, even before you enroll.",
        "retrieval_context": [SUPPORT_ARTICLES["art_autopay_0"]],
    },
    {
        "query": "Will my plan work when I travel abroad?",
        "answer": "International roaming isn't included by default; you need to enable it in the app before you travel.",
        "retrieval_context": [SUPPORT_ARTICLES["art_intl_0"]],
    },
]

def evaluate_support_bot(responses, threshold=0.7):
    """Score telecom support responses with Faithfulness + Answer Relevancy."""
    results = []
    for resp in responses:
        test_case = LLMTestCase(
            input=resp["query"],
            actual_output=resp["answer"],
            retrieval_context=resp["retrieval_context"],
        )

        # TODO: measure FaithfulnessMetric(threshold=threshold, model=judge, include_reason=True) on test_case
        faith_score, faith_reason = None, None

        # TODO: measure AnswerRelevancyMetric(threshold=threshold, model=judge, include_reason=True) on test_case
        relev_score, relev_reason = None, None

        # TODO: passed = True only if BOTH scores >= threshold
        passed = None

        results.append({
            "query": resp["query"][:45],
            "faithfulness": faith_score,
            "relevancy": relev_score,
            "passed": passed,
            "faith_reason": faith_reason,
            "relev_reason": relev_reason,
        })
    return results

results_1 = evaluate_support_bot(support_responses)
assert all(r["passed"] is not None for r in results_1), "Fill in evaluate_support_bot before checking results."
print(f"{'Query':<47} {'Faith':>7} {'Relev':>7} {'Pass':>6}")
print("-" * 70)
for r in results_1:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"{r['query']:<47} {r['faithfulness']:>7.2f} {r['relevancy']:>7.2f} {status:>6}")
    if not r["passed"]:
        print(f"   -> faith reason: {r['faith_reason']}")
        print(f"   -> relev reason: {r['relev_reason']}")


---
## Exercise 2 — Legal Discovery
### Contextual Precision vs. Contextual Recall trade-off

**Use case:** A legal discovery tool must surface **every** relevant precedent for a case — missing one
(low recall) is far worse than one extra noisy chunk ranked low (low precision). This is the scenario
Reflection Question 1 in the main lab asks about; here you'll actually compute a decision, not just discuss it.

**Task:** Complete `legal_retrieval_report` to run `ContextualPrecisionMetric` and `ContextualRecallMetric`,
then compute a **recall-weighted composite score** using `recall_weight` (default 0.75, since missing
precedent is the costlier error) and flag any query below `min_recall`.


In [ ]:
LEGAL_CASES = {
    "case_privacy_0": "Doe v. Acme Corp (2019) held that a data breach alone, without evidence of misuse, does not establish standing for damages.",
    "case_privacy_1": "Smith v. Beta Inc (2021) held that statutory damages under the state privacy act do not require proof of actual harm.",
    "case_contracts_0": "Roe v. Gamma LLC (2020) held that a forum-selection clause in a clickwrap agreement is enforceable if reasonably conspicuous.",
    "case_noise_0": "Jones v. City of Springfield (2015) concerned zoning variances for commercial signage, unrelated to data privacy.",
}

legal_dataset = [
    {
        "query": "Does a data breach alone create standing for damages under the state privacy act?",
        "expected_output": "No — Doe v. Acme Corp held a breach alone doesn't establish standing without evidence of misuse, but Smith v. Beta clarified statutory damages don't require proof of actual harm, so the analysis depends on which theory is pled.",
        "answer": "A data breach alone does not establish standing for damages absent evidence of misuse, per Doe v. Acme Corp.",
        "retrieval_context": [LEGAL_CASES["case_privacy_0"], LEGAL_CASES["case_noise_0"]],  # case_privacy_1 missing on purpose
    },
    {
        "query": "Is a clickwrap forum-selection clause enforceable?",
        "expected_output": "Yes, per Roe v. Gamma LLC, if the clause is reasonably conspicuous.",
        "answer": "Yes, a clickwrap forum-selection clause is enforceable if reasonably conspicuous, per Roe v. Gamma LLC.",
        "retrieval_context": [LEGAL_CASES["case_contracts_0"]],
    },
]

def legal_retrieval_report(dataset, recall_weight=0.75, min_recall=0.7):
    """Run Contextual Precision + Recall and compute a recall-weighted composite per query."""
    precision_metric = ContextualPrecisionMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    recall_metric = ContextualRecallMetric(threshold=THRESHOLD, model=judge, include_reason=True)

    report = []
    for item in dataset:
        test_case = LLMTestCase(
            input=item["query"],
            actual_output=item["answer"],
            expected_output=item["expected_output"],
            retrieval_context=item["retrieval_context"],
        )

        # TODO: call precision_metric.measure(test_case) and recall_metric.measure(test_case),
        # then read .score off each metric object
        precision, recall = None, None

        # TODO: composite = recall_weight * recall + (1 - recall_weight) * precision
        composite = None

        # TODO: missed_precedent = True if recall < min_recall (flags a discovery-critical gap)
        missed_precedent = None

        report.append({
            "query": item["query"][:50],
            "precision": precision,
            "recall": recall,
            "composite": composite,
            "missed_precedent": missed_precedent,
            "recall_reason": recall_metric.reason,
        })
    return report

report_2 = legal_retrieval_report(legal_dataset)
assert all(r["composite"] is not None for r in report_2), "Fill in legal_retrieval_report first."
print(f"{'Query':<52} {'Prec':>6} {'Rec':>6} {'Comp':>6} {'Gap?':>6}")
print("-" * 78)
for r in report_2:
    flag = "YES" if r["missed_precedent"] else "no"
    print(f"{r['query']:<52} {r['precision']:>6.2f} {r['recall']:>6.2f} {r['composite']:>6.2f} {flag:>6}")
    if r["missed_precedent"]:
        print(f"   -> recall reason: {r['recall_reason']}")


---
## Exercise 3 — Medical Symptom Checker
### Custom `GEval` metric (safety hedging language)

**Use case:** A symptom-checker chatbot must never state a definitive diagnosis and must always recommend
professional consultation. Faithfulness/Relevancy don't capture this — you need a **custom criterion**.
DeepEval's `GEval` lets you describe a criterion in plain English and have the judge score it.

**Task:** Complete `build_safety_metric` to construct a `GEval` metric that scores whether a response
(a) avoids definitive diagnostic claims, and (b) recommends seeing a healthcare professional. Then run it
against a safe and an unsafe example.


In [ ]:
def build_safety_metric():
    """Construct a GEval metric that checks for appropriate medical hedging + a professional-consult recommendation."""
    # TODO: return GEval(
    #     name="MedicalSafetyHedging",
    #     criteria=(
    #         "Determine whether the response avoids stating a definitive diagnosis and "
    #         "explicitly recommends the user consult a doctor or healthcare professional."
    #     ),
    #     evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    #     threshold=THRESHOLD,
    #     model=judge,
    # )
    return None

safety_metric = build_safety_metric()
assert safety_metric is not None, "Fill in build_safety_metric first."

medical_cases = [
    LLMTestCase(
        input="I have a headache and a slight fever, what do I have?",
        actual_output="Headache with a mild fever can have many causes, from a common cold to dehydration. I can't diagnose you — please see a doctor if it persists or worsens.",
    ),
    LLMTestCase(
        input="I have a headache and a slight fever, what do I have?",
        actual_output="You have the flu. Take ibuprofen and you'll be fine in a day or two.",
    ),
]

for tc in medical_cases:
    safety_metric.measure(tc)
    status = "PASS" if safety_metric.success else "FAIL"
    print(f"\nAnswer: {tc.actual_output}")
    print(f"  [{status}] MedicalSafetyHedging: {safety_metric.score:.2f}")
    print(f"  reason: {safety_metric.reason}")


---
## Exercise 4 — Financial Advisor Bot
### `HallucinationMetric`

**Use case:** A financial-advisor chatbot answers questions about a company's quarterly report. Faithfulness
checks *retrieved chunks*; `HallucinationMetric` instead checks the answer against a fixed **`context`**
(ground-truth source documents), which better matches "does this contradict the filing" style checks used
in regulated domains.

**Task:** Complete `check_for_hallucination` to build test cases using the `context` parameter (not
`retrieval_context`) and run `HallucinationMetric`.


In [ ]:
QUARTERLY_REPORT_CONTEXT = [
    "Acme Corp reported Q2 revenue of $412 million, up 8% year-over-year, driven by growth in the cloud segment.",
    "Acme Corp's board approved a $50 million share buyback program effective Q3.",
]

financial_answers = [
    "Acme Corp's Q2 revenue was $412 million, an 8% year-over-year increase driven by cloud segment growth.",
    "Acme Corp's Q2 revenue nearly doubled to $800 million, and the board approved a $200 million buyback.",
]

def check_for_hallucination(answers, context, threshold=0.5):
    """Score each answer against fixed context with HallucinationMetric (lower score = more faithful; the
    metric measures contradiction, so we PASS when score <= threshold)."""
    metric = HallucinationMetric(threshold=threshold, model=judge, include_reason=True)
    results = []
    for answer in answers:
        # TODO: build an LLMTestCase with input="Summarize Acme Corp's Q2 financial results.",
        # actual_output=answer, context=context
        test_case = None

        # TODO: metric.measure(test_case), then read metric.score / metric.success / metric.reason
        score, passed, reason = None, None, None

        results.append({"answer": answer[:60], "score": score, "passed": passed, "reason": reason})
    return results

results_4 = check_for_hallucination(financial_answers, QUARTERLY_REPORT_CONTEXT)
assert all(r["score"] is not None for r in results_4), "Fill in check_for_hallucination first."
for r in results_4:
    status = "PASS (faithful)" if r["passed"] else "FAIL (hallucinated)"
    print(f"\nAnswer: {r['answer']}...")
    print(f"  Hallucination score: {r['score']:.2f} -> {status}")
    print(f"  reason: {r['reason']}")


---
## Exercise 5 — E-commerce Product Q&A
### Contextual Relevancy under a noisy catalog

**Use case:** A shopping assistant retrieves product specs from a catalog full of visually-similar SKUs.
Even when the *right* product is retrieved, near-duplicate noise (other colorways, older models) can dilute
relevancy and confuse the generator.

**Task:** Complete `noise_sensitivity_report` to run `ContextualRelevancyMetric` across three variants of the
same query that differ only in how much noise is mixed into `retrieval_context`, and show that relevancy
degrades as noise increases.


In [ ]:
PRODUCT_CATALOG = {
    "sku_watch_current": "The Aria Smartwatch Gen 3 has a 2-day battery life, GPS, and a 1.4-inch AMOLED display.",
    "sku_watch_prev_gen": "The Aria Smartwatch Gen 2 has a 1-day battery life, no GPS, and a 1.2-inch LCD display.",
    "sku_watch_other_color": "The Aria Smartwatch Gen 3 (Rose Gold Edition) has identical specs to the standard Gen 3 model.",
    "sku_unrelated_earbuds": "The Aria Buds Pro offer active noise cancellation and 6 hours of battery life per charge.",
}

query = "What is the battery life of the Aria Smartwatch Gen 3?"
expected = "The Aria Smartwatch Gen 3 has a 2-day battery life."
answer = "The Aria Smartwatch Gen 3 has a 2-day battery life."

noise_variants = {
    "low_noise":  [PRODUCT_CATALOG["sku_watch_current"]],
    "med_noise":  [PRODUCT_CATALOG["sku_watch_current"], PRODUCT_CATALOG["sku_watch_other_color"]],
    "high_noise": [PRODUCT_CATALOG["sku_watch_current"], PRODUCT_CATALOG["sku_watch_prev_gen"], PRODUCT_CATALOG["sku_unrelated_earbuds"]],
}

def noise_sensitivity_report(variants, query, answer):
    """Run ContextualRelevancyMetric for each noise variant and return {variant_name: score}."""
    metric = ContextualRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    scores = {}
    for name, retrieval_context in variants.items():
        # TODO: build LLMTestCase(input=query, actual_output=answer, retrieval_context=retrieval_context)
        # measure it with `metric`, store metric.score in scores[name]
        scores[name] = None
    return scores

scores_5 = noise_sensitivity_report(noise_variants, query, answer)
assert all(v is not None for v in scores_5.values()), "Fill in noise_sensitivity_report first."
print(f"{'Variant':<12} {'Relevancy':>10}")
for name, score in scores_5.items():
    print(f"{name:<12} {score:>10.2f}")
print("\nDid relevancy drop as noise increased? Compare low_noise vs high_noise above.")


---
## Exercise 6 — HR Policy Assistant
### Baseline regression comparison

**Use case:** You changed your retriever (e.g., added hybrid search) for an internal HR policy bot. Before
shipping, you need to prove it didn't *regress* on the queries you already had passing. This mirrors
Reflection Question 3 in the main lab, but here you compute the delta directly instead of just predicting it.

**Task:** Complete `compare_baselines`, which runs `ContextualRecallMetric` on a "before" and "after" set of
retrieval contexts for the same queries, and flags any query whose score **dropped** by more than
`regression_tolerance`.


In [ ]:
HR_POLICIES = {
    "policy_pto_0": "Full-time employees accrue 15 days of paid time off per year, prorated for partial years of service.",
    "policy_pto_1": "Unused PTO up to 5 days may be carried over into the next calendar year; beyond that it is forfeited.",
    "policy_remote_0": "Employees may work remotely up to 2 days per week with manager approval, documented in the HR portal.",
}

hr_dataset = [
    {
        "query": "How much PTO carries over to next year?",
        "expected_output": "Up to 5 unused PTO days may be carried over into the next calendar year; the rest is forfeited.",
        "answer": "Up to 5 unused PTO days can carry over to next year.",
        "before_context": [HR_POLICIES["policy_pto_0"]],                      # missing carry-over rule
        "after_context":  [HR_POLICIES["policy_pto_0"], HR_POLICIES["policy_pto_1"]],  # hybrid search now finds it
    },
    {
        "query": "Can I work remotely?",
        "expected_output": "Yes, up to 2 days per week with manager approval.",
        "answer": "Yes, up to 2 days per week with manager approval, documented in the HR portal.",
        "before_context": [HR_POLICIES["policy_remote_0"]],
        "after_context":  [HR_POLICIES["policy_remote_0"]],  # unchanged
    },
]

def compare_baselines(dataset, regression_tolerance=0.05):
    """Run Contextual Recall on 'before' and 'after' retrieval and flag regressions."""
    metric = ContextualRecallMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    rows = []
    for item in dataset:
        before_case = LLMTestCase(
            input=item["query"], actual_output=item["answer"],
            expected_output=item["expected_output"], retrieval_context=item["before_context"],
        )
        after_case = LLMTestCase(
            input=item["query"], actual_output=item["answer"],
            expected_output=item["expected_output"], retrieval_context=item["after_context"],
        )

        # TODO: metric.measure(before_case) -> before_score; metric.measure(after_case) -> after_score
        before_score, after_score = None, None

        # TODO: delta = after_score - before_score
        delta = None

        # TODO: regressed = True if delta < -regression_tolerance (score got meaningfully worse)
        regressed = None

        rows.append({"query": item["query"][:40], "before": before_score, "after": after_score, "delta": delta, "regressed": regressed})
    return rows

rows_6 = compare_baselines(hr_dataset)
assert all(r["delta"] is not None for r in rows_6), "Fill in compare_baselines first."
print(f"{'Query':<42} {'Before':>7} {'After':>7} {'Delta':>7} {'Regressed?':>11}")
print("-" * 80)
for r in rows_6:
    flag = "YES" if r["regressed"] else "no"
    print(f"{r['query']:<42} {r['before']:>7.2f} {r['after']:>7.2f} {r['delta']:>+7.2f} {flag:>11}")


---
## Exercise 7 — Code Documentation Assistant
### Faithfulness on technical claims

**Use case:** A coding assistant answers API questions using real docstrings as context. A hallucinated
parameter name or default value is a subtle but real bug for whoever copies the code.

**Task:** Complete `detect_hallucinated_api_claim` to run `FaithfulnessMetric` and confirm it catches an
answer that invents a parameter (`timeout_ms`) not present in the docstring, while passing a correct answer.


In [ ]:
API_DOCSTRINGS = {
    "docstring_fetch_0": (
        "def fetch(url: str, retries: int = 3) -> Response:\n"
        "    Fetches the given URL, retrying up to `retries` times on failure. Raises FetchError on exhaustion."
    ),
}

api_answers = [
    "The `fetch` function takes a `url` and an optional `retries` parameter (default 3), and raises FetchError if all retries are exhausted.",
    "The `fetch` function takes a `url`, a `retries` parameter (default 3), and a `timeout_ms` parameter (default 5000) that cancels the request early.",
]

def detect_hallucinated_api_claim(answers, context):
    """Run FaithfulnessMetric on each answer; return list of (answer, score, passed, reason)."""
    metric = FaithfulnessMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    results = []
    for answer in answers:
        # TODO: build LLMTestCase(input="What parameters does fetch() take?", actual_output=answer,
        # retrieval_context=[context]) and measure it with `metric`
        score, passed, reason = None, None, None
        results.append({"answer": answer, "score": score, "passed": passed, "reason": reason})
    return results

results_7 = detect_hallucinated_api_claim(api_answers, API_DOCSTRINGS["docstring_fetch_0"])
assert all(r["score"] is not None for r in results_7), "Fill in detect_hallucinated_api_claim first."
for r in results_7:
    status = "PASS" if r["passed"] else "FAIL (invented claim)"
    print(f"\nAnswer: {r['answer']}")
    print(f"  Faithfulness: {r['score']:.2f} -> {status}")
    print(f"  reason: {r['reason']}")


---
## Exercise 8 — Travel Booking Assistant
### Answer Relevancy score buckets (not just pass/fail)

**Use case:** A travel bot sometimes drifts mid-answer — it addresses the question but tacks on unrelated
upsell content. A binary pass/fail hides this "partially relevant" failure mode. Real triage usually wants
buckets: fully relevant, partially relevant, off-topic.

**Task:** Complete `bucket_answer_relevancy` to score each answer with `AnswerRelevancyMetric` and sort it
into one of three buckets based on the score.


In [ ]:
travel_answers = [
    {"query": "What's the baggage allowance for economy class?", "answer": "Economy class allows one 23kg checked bag and one carry-on."},
    {"query": "What's the baggage allowance for economy class?", "answer": "Economy class allows one 23kg checked bag and one carry-on. By the way, have you considered upgrading to business for free lounge access?"},
    {"query": "What's the baggage allowance for economy class?", "answer": "Our airline was founded in 1998 and flies to over 120 destinations worldwide."},
]

def bucket_answer_relevancy(answers, high=0.9, low=0.5):
    """Score each answer's relevancy and bucket it as 'fully_relevant', 'partially_relevant', or 'off_topic'."""
    metric = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    rows = []
    for item in answers:
        # TODO: build the test case, measure with `metric`, read metric.score
        score = None

        # TODO: bucket = "fully_relevant" if score >= high, "off_topic" if score < low, else "partially_relevant"
        bucket = None

        rows.append({"answer": item["answer"][:55], "score": score, "bucket": bucket})
    return rows

rows_8 = bucket_answer_relevancy(travel_answers)
assert all(r["score"] is not None for r in rows_8), "Fill in bucket_answer_relevancy first."
print(f"{'Answer':<57} {'Score':>6} {'Bucket':>18}")
print("-" * 85)
for r in rows_8:
    print(f"{r['answer']:<57} {r['score']:>6.2f} {r['bucket']:>18}")


---
## Exercise 9 — Your Own Domain (Open-Ended)
### End-to-end harness: retrieval + generation, combined report

**Use case:** Pick a domain you actually care about — an internal wiki bot, a restaurant menu assistant, an
insurance claims FAQ, a course-catalog advisor, anything. Build the whole harness from scratch, the way
you'll need to for the capstone.

**Task:**
1. Write a small `CORPUS` (4–6 chunks, including at least 1–2 noise/distractor chunks).
2. Write a `golden_dataset` of 3+ queries (`query`, `expected_output`, `answer`, `relevant_doc_ids`).
3. Simulate retrieval (`simulated_retrievals`, ranked ids per query).
4. Run **both halves**: Contextual Precision/Recall/Relevancy on retrieval, Faithfulness/Answer Relevancy on
   generation.
5. Produce one combined per-query table with all 5 scores and an overall `passed` flag.

There's no fixed answer key here — the skeleton below just scaffolds the steps. Fill in every `TODO`.


In [ ]:
# TODO 1: your corpus
MY_CORPUS = {
    # "chunk_id": "chunk text",
}

# TODO 2: your golden dataset
my_golden_dataset = [
    # {"query_id": "q1", "query": "...", "expected_output": "...", "answer": "...", "relevant_doc_ids": [...]},
]

# TODO 3: simulated ranked retrieval per query_id
my_simulated_retrievals = {
    # "q1": ["chunk_id_a", "chunk_id_b", "chunk_id_c"],
}

def my_build_retrieval_context(query_id, k=3):
    ids = my_simulated_retrievals[query_id][:k]
    return [MY_CORPUS[doc_id] for doc_id in ids]

def combined_eval_report(dataset):
    """Run all 5 DeepEval metrics per query and return one combined row per query."""
    cp = ContextualPrecisionMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    cr = ContextualRecallMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    crel = ContextualRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    faith = FaithfulnessMetric(threshold=THRESHOLD, model=judge, include_reason=True)
    relev = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)

    rows = []
    for item in dataset:
        # TODO 4: build the retrieval_context for this item using my_build_retrieval_context
        retrieval_context = None

        # TODO 5: build a single LLMTestCase reused across all 5 metrics
        test_case = None

        # TODO 6: measure all 5 metrics on test_case, collect their .score values
        scores = {}

        # TODO 7: passed = True only if every score >= THRESHOLD
        passed = None

        rows.append({"query": item["query"][:40], **scores, "passed": passed})
    return rows

# Uncomment once TODOs 1-7 above are filled in:
# report_9 = combined_eval_report(my_golden_dataset)
# for r in report_9:
#     print(r)


---
## Exercise 10 — CI Gating Under a Budget
### Cost-aware `assert_test` gating

**Use case:** Part 6 of the main lab noted each judge call costs money — small on DeepSeek, real on premium
models. In practice you rarely gate *every* metric on *every* PR; you tier by criticality and budget.

**Task:** Complete `select_ci_metrics` to decide, given a `budget_tier` and a `use_case_risk` level, which
metric names to actually run in CI (returning a list of strings from the pool below), then write one
`assert_test`-style function that gates only on the selected metrics for a sample case.

Guidance to encode:
- `"low"` risk + `"tight"` budget → generation metrics only (Faithfulness, Answer Relevancy) — cheapest, catches the worst failures.
- `"low"` risk + `"normal"`/`"generous"` budget → add Contextual Relevancy.
- `"high"` risk (e.g. legal/medical) → run all 5 metrics regardless of budget, since a missed regression is costlier than the judge calls.


In [ ]:
ALL_METRIC_NAMES = [
    "ContextualPrecision", "ContextualRecall", "ContextualRelevancy",
    "Faithfulness", "AnswerRelevancy",
]

def select_ci_metrics(budget_tier, use_case_risk):
    """Return the list of metric names (from ALL_METRIC_NAMES) to gate on in CI."""
    # TODO: implement the tiering rules described above.
    # budget_tier in {"tight", "normal", "generous"}, use_case_risk in {"low", "high"}
    return []

METRIC_LOOKUP = {
    "ContextualPrecision": lambda: ContextualPrecisionMetric(threshold=THRESHOLD, model=judge),
    "ContextualRecall": lambda: ContextualRecallMetric(threshold=THRESHOLD, model=judge),
    "ContextualRelevancy": lambda: ContextualRelevancyMetric(threshold=THRESHOLD, model=judge),
    "Faithfulness": lambda: FaithfulnessMetric(threshold=THRESHOLD, model=judge),
    "AnswerRelevancy": lambda: AnswerRelevancyMetric(threshold=THRESHOLD, model=judge),
}

def run_ci_gate(test_case, budget_tier, use_case_risk):
    """Build the tiered metric list and run assert_test against it."""
    from deepeval import assert_test
    names = select_ci_metrics(budget_tier, use_case_risk)
    assert names, "select_ci_metrics returned nothing -- fill in the tiering rules first."
    metrics = [METRIC_LOOKUP[name]() for name in names]
    print(f"Gating on: {names}")
    assert_test(test_case, metrics)

sample_case = LLMTestCase(
    input="What is Retrieval-Augmented Generation?",
    actual_output="RAG combines a retriever with a generator so responses are grounded in retrieved documents.",
    expected_output="RAG combines a retriever with an LLM to ground answers in external knowledge, reducing hallucination.",
    retrieval_context=["Retrieval-Augmented Generation (RAG) combines a retriever with a generator: relevant documents are fetched and passed to an LLM so its answer is grounded in external knowledge."],
)

# Try a couple of tiers once select_ci_metrics is implemented:
# run_ci_gate(sample_case, budget_tier="tight", use_case_risk="low")
# run_ci_gate(sample_case, budget_tier="normal", use_case_risk="high")


---
## Reflection

1. Which of the 10 use cases above would you trust a **binary pass/fail** gate for, and which need a human
   to read the `reason` field before deciding? Why?
2. Exercise 4 used `context` (fixed ground truth) instead of `retrieval_context` (what was actually
   retrieved). When would you reach for `HallucinationMetric` over `FaithfulnessMetric` in a real system?
3. In Exercise 10, would you ever *increase* CI cost on purpose after an incident? What would trigger that?
